# Network Intrusion Detection: ABC vs ACO feature selection and Random Forest tuning
Bio-Inspired Algorithms AAT (23DS7BSBIO)

This notebook walks through the pipeline end to end: data, baseline, the two nature-inspired optimisers
(implemented from scratch in `src/`), a small live demo, and the final results of the full experiment
(15 seeds, produced by `run_experiments.py`).

In [ ]:
import json, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd()))   # run the notebook from the project root

from src.preprocess import load_nsl_kdd, HOLDOUT_ATTACKS
from src.baselines import feature_mi
from src.evaluate import evaluate_model, BASELINE_PARAMS
from src.fitness import FitnessEvaluator
from src.abc import ArtificialBeeColony
from src.aco import AntColony
from src.stats import significance_table

pd.set_option("display.width", 200)

## 1. Dataset and problem definition
NSL-KDD: every row is a network connection with 41 features (3 categorical). Task: **binary classification,
normal vs attack**. `KDDTest+` contains attack types that never appear in training, so it measures
generalisation, not just memorisation.

Preprocessing (`src/preprocess.py`): drop duplicates, one-hot encode `protocol_type`/`service`/`flag`
(41 -> 122 features), MinMax-scale (fitted on train only). The optimiser is validated on **held-out attack types**
(listed below) so that its fitness reflects detection of unseen attacks. `KDDTest+` is used only for final scores.

In [ ]:
data = load_nsl_kdd("data", val_mode="attack_holdout")
print("features:", data.n_features)
print("train / val / test rows:", len(data.y_train), len(data.y_val), len(data.y_test))
print("attack share  train / val / test:", data.y_train.mean().round(3), data.y_val.mean().round(3), data.y_test.mean().round(3))
print("attack types held out for validation:", HOLDOUT_ATTACKS)

## 2. Baseline model
A Random Forest on all 122 features with default settings (100 trees, unlimited depth). RF is a good fit here: it handles
mixed, unscaled features, is robust to noise and trains quickly enough to be called hundreds of times inside an optimiser.

In [ ]:
baseline = evaluate_model(np.ones(data.n_features, dtype=bool), BASELINE_PARAMS, data, seed=0)
pd.Series({k: round(v, 4) for k, v in baseline.items() if k != "confusion"}, name="baseline RF (test set)")

## 3. Nature-inspired optimisation
Both algorithms search the same space: a **binary mask over the 122 features + three RF hyper-parameters**
(`n_estimators`, `max_depth`, `min_samples_split`), scored by one shared fitness function

`fitness = 0.8 * F1(validation) + 0.1 * (1 - features_used / 122) + 0.1 * (1 - n_estimators / 100)`

| | ABC (`src/abc.py`) | ACO (`src/aco.py`) |
|---|---|---|
| Candidate | food source (continuous position, sigmoid to a 0/1 mask) | ant building a subset feature by feature |
| Variation | neighbour move `v = x + phi (x - x_k)` | probabilistic choice `~ tau^alpha * eta^beta` (eta = mutual information) |
| Selection | greedy per source; onlookers pick sources by roulette wheel | iteration-best and global-best ants deposit pheromone |
| Diversity | scout bee replaces a stagnant source after `limit` failures | evaporation `(1 - rho)` and pheromone bounds (MAX-MIN) |

Below is a short live demo (small budget so it runs in under a minute). The real experiment uses 300 evaluations x 15 seeds.

In [ ]:
BUDGET, POP, SEED = 40, 6, 0
mi = feature_mi(data)
new_ev = lambda: FitnessEvaluator(data, train_subsample=8000, val_subsample=6000, seed=SEED)

abc_res = ArtificialBeeColony(new_ev(), n_sources=POP, limit=POP, max_evals=BUDGET, seed=SEED).run()
aco_res = AntColony(new_ev(), mi, n_ants=POP, max_evals=BUDGET, seed=SEED).run()

for name, res in (("ABC", abc_res), ("ACO", aco_res)):
    m = evaluate_model(res["best_mask"], res["best_params"], data, seed=SEED)
    print(f"{name}: fitness={res['best_fitness']:.3f} | features={m['n_features']} | params={res['best_params']} | "
          f"test F1={m['f1']:.3f} acc={m['accuracy']:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
for name, res, c in (("ABC", abc_res, "#1f77b4"), ("ACO", aco_res, "#d62728")):
    h = np.array(res["history"])
    ax.step(h[:, 0], h[:, 1], where="post", label=name, color=c)
ax.set(xlabel="fitness evaluations", ylabel="best fitness", title="Demo convergence (tiny budget)")
ax.legend(); ax.grid(alpha=.3); plt.show()

## 4. Full experiment: results
Produced by `python run_experiments.py --seeds 15 --max-evals 300 --pop 10`. Six models, 15 independent seeds,
all scored on `KDDTest+`. **Baseline RF** and **MI top-30** are references; **ABC-RF / ACO-RF** select features *and* tune the RF;
the **(tune only)** variants keep all 122 features and tune only the hyper-parameters, which separates the two effects.

In [ ]:
RES = Path("results_15seeds")
summary = pd.read_csv(RES / "summary.csv", index_col=0)
cols = ["accuracy", "precision", "recall", "f1", "auc", "n_features"]
pd.DataFrame({c: summary[c + "_mean"].round(4).astype(str) + " +/- " + summary[c + "_std"].round(4).astype(str) for c in cols})

In [ ]:
# Timing is reported as the median: a few runs hit system slowdowns that distort mean/std.
from src.evaluate import runs_to_frame
runs = json.loads((RES / "results.json").read_text())["runs"]
df = runs_to_frame(runs)
timing = df.groupby("model", sort=False)[["train_time", "predict_time"]].median().round(3)
timing["train_speedup"] = (timing.loc["Baseline RF", "train_time"] / timing["train_time"]).round(1)
timing["inference_speedup"] = (timing.loc["Baseline RF", "predict_time"] / timing["predict_time"]).round(1)
timing

In [ ]:
# Paired tests against the baseline (same seeds): is any difference larger than run-to-run noise?
sig = significance_table(runs)
sig[sig.metric == "f1"].round(4).reset_index(drop=True)

In [ ]:
for name in ("convergence", "metrics_comparison", "features_kept", "time_comparison", "feature_frequency", "confusion_matrices"):
    display(Image(filename=str(RES / "figures" / f"{name}.png"), width=720))

## 5. Ablation: what the validation split changes
With a random validation split a plain Random Forest scores F1 = 0.998 on validation but only 0.768 on `KDDTest+`, so the
optimiser cannot see the generalisation gap and rewards smaller models at the expense of test F1. Re-running with held-out attack types fixes that.
(`results_random_val/` holds the random-split run.)

In [ ]:
def f1_table(folder):
    s = pd.read_csv(Path(folder) / "summary.csv", index_col=0)
    return s[["f1_mean", "f1_std", "n_features_mean"]].round(4)

pd.concat({"random validation": f1_table("results_random_val"),
           "attack-holdout validation": f1_table("results_15seeds").loc[["Baseline RF", "MI top-30", "ABC-RF", "ACO-RF"]]})

## 6. Reproducing everything
```bash
python run_experiments.py --seeds 15 --max-evals 300 --pop 10 --out results_15seeds
python run_experiments.py --seeds 15 --max-evals 300 --pop 10 --val-mode random --skip-tune-only --out results_random_val
python -m pytest tests -q
```
Fitness values, seeds and subsamples are deterministic, so the selected features and the scores repeat exactly
(only wall-clock timings vary).